# qlora-lab on a free Colab T4: train + compare in one run

**Before running**: Runtime -> Change runtime type -> **T4 GPU**.

This notebook is self-contained: it clones the repo, generates data, measures
the raw base model, runs the QLoRA fine-tune, re-measures, and prints the
base-vs-tuned table. Total wall time on a T4 is roughly 30-50 minutes, most of
it the ~3-epoch training run.

It evaluates with Unsloth inference directly instead of a vLLM server, because
running a vLLM server inside a T4 Colab is fragile. The numbers are the same
kind: schema validity, exact match, latency, tokens. vLLM serving (notebook 04)
is for when you deploy.

In [ ]:
# 1. GPU stack. unsloth pulls trl/peft/transformers/datasets/bitsandbytes.
%pip install -q unsloth

In [ ]:
# 2. Code + data
!git clone https://github.com/zyziyun/qlora-lab.git
%cd qlora-lab
!python scripts/make_data.py --n 800
import sys; sys.path.insert(0, 'src')

In [ ]:
# 3. Eval helper: run any (model, tokenizer) over the test set and build
#    Prediction objects so the repo's evaluate harness works unchanged.
import time, torch
from qlora_lab import dataset as qds, evaluate as qev
from qlora_lab.dataset import SYSTEM_PROMPT
from qlora_lab.predict import Prediction

test = qds.read_jsonl('data/test.jsonl')

def run_eval(model, tokenizer, test, max_new_tokens=96):
    preds = []
    for e in test:
        msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': e['message']}]
        # enable_thinking=False matters for Qwen3: thinking mode would emit a
        # long <think> block before the JSON and wreck latency and token counts.
        inputs = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors='pt',
            enable_thinking=False).to(model.device)
        t0 = time.perf_counter()
        out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
        dt = time.perf_counter() - t0
        gen = out[0][inputs.shape[1]:]
        preds.append(Prediction(
            raw=tokenizer.decode(gen, skip_special_tokens=True),
            latency_s=dt,
            prompt_tokens=int(inputs.shape[1]),
            completion_tokens=int(gen.shape[0])))
    return preds

In [ ]:
# 4. Baseline: the raw 4-bit base, prompted. ~5 min for 100 examples.
from unsloth import FastLanguageModel
BASE = 'unsloth/Qwen3-8B-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
base_preds = run_eval(model, tokenizer, test)
rb = qev.evaluate(base_preds, test, in_price=0.05e-6, out_price=0.20e-6)
print('BASE :', rb.summary())
print('sample failures:', [f['reason'] for f in rb.failures[:5]])
# free the GPU before training loads its own copy
del model, tokenizer
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. QLoRA fine-tune. ~20-40 min on a T4 for 573 examples x 3 epochs.
from qlora_lab import train as qtrain
cfg = qtrain.TrainConfig(base_model=BASE, output_dir='outputs/adapter')
adapter_dir = qtrain.train('data/train.jsonl', cfg)
print('adapter saved to', adapter_dir)
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 6. Re-measure with the adapter and print the table that matters.
model, tokenizer = FastLanguageModel.from_pretrained(
    'outputs/adapter', max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tuned_preds = run_eval(model, tokenizer, test)
rt = qev.evaluate(tuned_preds, test, in_price=0.05e-6, out_price=0.20e-6)
print('TUNED:', rt.summary())
print()
print(qev.compare(rb, rt))

If tuned does not clear base by 10%+ on schema validity, suspect the fine-tune,
not the model: check the loss mask and data formatting in notebook 02.

To benchmark more bases (the resume's *benchmarking Qwen3, LLaMA3, Gemma*),
rerun cells 4-6 with `BASE` set to e.g. `unsloth/llama-3.1-8b-bnb-4bit` or
`unsloth/gemma-2-9b-bnb-4bit` and keep the per-base tables.

In [ ]:
# 7. Take the adapter home (~tens of MB) for vLLM serving (notebook 04).
!zip -qr adapter.zip outputs/adapter
from google.colab import files
files.download('adapter.zip')